In [3]:
# Fine-tune ERNIE for 4-class Chinese domain classification — no Trainer/Accelerate deps
import json, os, random
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm import tqdm

from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    DataCollatorWithPadding, set_seed
)
try:
    from transformers.optimization import get_linear_schedule_with_warmup
except Exception:
    get_linear_schedule_with_warmup = None  # fallback below

# ---------------- Config ----------------
CHECKPOINT = "nghuyong/ernie-3.0-base-zh"
DATA_PATH = "wmt2425_en_zh_CN.jsonl"
TEXT_FIELD = "target"           # Chinese text
LABEL_FIELD = "domain"          # "news" | "social" | "speech" | "literary"
LABELS = ["news", "social", "speech", "literary"]

MAX_LEN = 256
SEED = 42
BATCH_TRAIN = 16
BATCH_EVAL = 32
LR = 2e-5
EPOCHS = 3
WARMUP_RATIO = 0.1
OUTPUT_DIR = "ernie-domain-clf"

set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------------- Data ----------------
with open(DATA_PATH, encoding="utf-8") as f:
    raw = [json.loads(line) for line in f]

label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for l, i in label2id.items()}

random.shuffle(raw)
n = len(raw)
n_train = int(0.8 * n)
n_val = int(0.1 * n)
train_raw = raw[:n_train]
val_raw   = raw[n_train:n_train + n_val]
test_raw  = raw[n_train + n_val:]

# ------------- Dataset ----------------
class ZhDomainDataset(Dataset):
    def __init__(self, rows, tokenizer, max_len, text_field, label_field, label2id):
        self.rows = rows
        self.tok = tokenizer
        self.max_len = max_len
        self.text_field = text_field
        self.label_field = label_field
        self.label2id = label2id
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, i):
        ex = self.rows[i]
        enc = self.tok(
            str(ex[self.text_field]),
            truncation=True,
            max_length=self.max_len,
            return_attention_mask=True
        )
        enc["labels"] = int(self.label2id[ex[self.label_field]])
        return enc

tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)
model = AutoModelForSequenceClassification.from_pretrained(
    CHECKPOINT, num_labels=len(LABELS), id2label=id2label, label2id=label2id
).to(device)

collator = DataCollatorWithPadding(tokenizer=tokenizer)
train_ds = ZhDomainDataset(train_raw, tokenizer, MAX_LEN, TEXT_FIELD, LABEL_FIELD, label2id)
val_ds   = ZhDomainDataset(val_raw,   tokenizer, MAX_LEN, TEXT_FIELD, LABEL_FIELD, label2id)
test_ds  = ZhDomainDataset(test_raw,  tokenizer, MAX_LEN, TEXT_FIELD, LABEL_FIELD, label2id)

train_loader = DataLoader(train_ds, batch_size=BATCH_TRAIN, shuffle=True,  collate_fn=collator)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_EVAL,  shuffle=False, collate_fn=collator)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_EVAL,  shuffle=False, collate_fn=collator)

# ------------- Optimizer/Scheduler -------------
optimizer = AdamW(model.parameters(), lr=LR)
total_steps = len(train_loader) * max(1, EPOCHS)
warmup_steps = int(WARMUP_RATIO * total_steps)

if get_linear_schedule_with_warmup is not None and total_steps > 0:
    scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
else:
    scheduler = None

scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

# ------------- Train/Eval helpers -------------
@torch.no_grad()
def evaluate(dataloader):
    model.eval()
    all_preds, all_labels = [], []
    for batch in dataloader:
        batch = {k: torch.as_tensor(v).to(device) for k, v in batch.items()}
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            outputs = model(**batch)
        logits = outputs.logits
        all_preds.append(logits.argmax(-1).detach().cpu().numpy())
        all_labels.append(batch["labels"].detach().cpu().numpy())
    y_pred = np.concatenate(all_preds)
    y_true = np.concatenate(all_labels)
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "f1_macro": float(f1_score(y_true, y_pred, average="macro")),
        "y_true": y_true,
        "y_pred": y_pred
    }

# ---------------- Training Loop ----------------
for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0

    # Wrap your DataLoader with tqdm for progress bar
    progress = tqdm(train_loader, desc=f"Epoch {epoch}", leave=False)
    for step, batch in enumerate(progress, 1):
        batch = {k: torch.as_tensor(v).to(device) for k, v in batch.items()}

        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            outputs = model(**batch)
            loss = outputs.loss

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        if scheduler is not None:
            scheduler.step()

        running_loss += loss.item()
        avg_loss = running_loss / step

        # update tqdm progress bar text dynamically
        progress.set_postfix(loss=f"{avg_loss:.4f}")

    # evaluate after each epoch
    val_metrics = evaluate(val_loader)
    print(f"Epoch {epoch} | Val acc {val_metrics['accuracy']:.4f} | Val F1 {val_metrics['f1_macro']:.4f}")


# ---------------- Test & Report ----------------
test_metrics = evaluate(test_loader)
print("Test accuracy:", test_metrics["accuracy"])
print("Test macro-F1:", test_metrics["f1_macro"])
print(classification_report(test_metrics["y_true"], test_metrics["y_pred"], target_names=LABELS, digits=4))

# ---------------- Save ----------------
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Saved model + tokenizer to {OUTPUT_DIR}")


Some weights of ErnieForSequenceClassification were not initialized from the model checkpoint at nghuyong/ernie-3.0-base-zh and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/var/folders/h0/55sj1y597v1c3hcdbxtl0d140000gn/T/ipykernel_13488/1398455540.py:101: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Epoch 1:   0%|          | 0/55 [00:00<?, ?it/s]/var/folders/h0/55sj1y597v1c3hcdbxtl0d14000

Epoch 1 | Val acc 0.8889 | Val F1 0.8557


Epoch 2:   0%|          | 0/55 [00:00<?, ?it/s]/var/folders/h0/55sj1y597v1c3hcdbxtl0d140000gn/T/ipykernel_13488/1398455540.py:135: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
/var/folders/h0/55sj1y597v1c3hcdbxtl0d140000gn/T/ipykernel_13488/1398455540.py:110: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):


Epoch 2 | Val acc 0.9444 | Val F1 0.9321


Epoch 3:   0%|          | 0/55 [00:00<?, ?it/s]/var/folders/h0/55sj1y597v1c3hcdbxtl0d140000gn/T/ipykernel_13488/1398455540.py:135: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
/var/folders/h0/55sj1y597v1c3hcdbxtl0d140000gn/T/ipykernel_13488/1398455540.py:110: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):


Epoch 3 | Val acc 0.9352 | Val F1 0.9187


/var/folders/h0/55sj1y597v1c3hcdbxtl0d140000gn/T/ipykernel_13488/1398455540.py:110: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):


Test accuracy: 0.963302752293578
Test macro-F1: 0.9518004896994259
              precision    recall  f1-score   support

        news     0.8000    1.0000    0.8889        12
      social     1.0000    0.9649    0.9821        57
      speech     0.9565    0.9167    0.9362        24
    literary     1.0000    1.0000    1.0000        16

    accuracy                         0.9633       109
   macro avg     0.9391    0.9704    0.9518       109
weighted avg     0.9684    0.9633    0.9644       109

Saved model + tokenizer to ernie-domain-clf
The OrderedVocab you are attempting to save contains holes for indices [12084], your vocabulary could be corrupted !
